# DWS 层 — 日粒度聚合

`fact_trips` → 按日 × 区域 × 来源 聚合，产出 `dws/trips_daily`。


## 1. SparkSession

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_date, year, month, dayofmonth

spark = (
    SparkSession.builder
    .appName("TLC-DWS-Build")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE     = os.path.expanduser("~/tlc-spark-pipeline")
DWD_FACT = f"{BASE}/data/warehouse/dwd/fact_trips"
DWS_OUT  = f"{BASE}/data/warehouse/dws/trips_daily"

TARGET_DATE = os.environ.get("TARGET_DATE", "2024-03-15")
yr  = int(TARGET_DATE.split("-")[0])
mo  = int(TARGET_DATE.split("-")[1])
day = int(TARGET_DATE.split("-")[2])
print(f"TARGET_DATE={TARGET_DATE}")

print(f"Spark {spark.version} ready")

26/10/19 14:57:23 WARN Utils: Your hostname, jelly-macbook.local resolves to a loopback address: 127.0.0.1; using 192.168.1.229 instead (on interface en0)
26/10/19 14:57:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/19 14:57:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


TARGET_DATE=2024-03-15
Spark 3.5.1 ready


## 2. 读取 fact_trips + 确认热点区域倾斜

In [2]:
fact = (
    spark.read.parquet(DWD_FACT)
    .filter((col("pickup_year") == yr) & (col("pickup_month") == mo) & (col("pickup_day") == day))
)
print(f"fact_trips: {fact.count():,} rows")

print("\n=== 热点区域（Top 10 上车量）===")
fact.groupBy("PULocationID") \
    .agg(F.count("*").alias("trip_count")) \
    .orderBy(F.desc("trip_count")) \
    .limit(10) \
    .show()

fact_trips: 862,909 rows

=== 热点区域（Top 10 上车量）===


+------------+----------+
|PULocationID|trip_count|
+------------+----------+
|         132|     16910|
|         138|     15570|
|          79|     13217|
|         161|     12637|
|         230|     12302|
|          68|     11168|
|         237|     11082|
|         170|     10897|
|         164|     10852|
|         236|     10794|
+------------+----------+



## 3. 聚合

In [ ]:
dws = (
    fact
    .withColumn("pickup_date", to_date("pickup_datetime"))
    .groupBy("pickup_date", "PULocationID", "source_type")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.sum("total_amount"), 2).alias("gmv"),
        F.round(F.sum("fare_amount"), 2).alias("fare_sum"),
        F.round(F.sum("tip_amount"), 2).alias("tip_sum"),
        F.round(F.sum("surcharge_total"), 2).alias("surcharge_sum"),
        F.round(F.sum("trip_distance"), 2).alias("distance_sum"),
        F.round(F.sum("trip_duration_min"), 2).alias("duration_sum"),
        # driver_pay: non-null only for fhv_hv records; used for platform take rate in ADS
        F.round(F.sum("driver_pay"), 2).alias("driver_pay_sum"),
    )
    .withColumn("pickup_year",  year("pickup_date"))
    .withColumn("pickup_month", month("pickup_date"))
    .withColumn("pickup_day",   dayofmonth("pickup_date"))
)

dws_count = dws.count()
print(f"DWS rows: {dws_count:,}")
dws.show(5)


## 5. 写出 DWS（动态分区覆盖）

In [5]:
%%time
(
    dws
    .write
    .partitionBy("pickup_year", "pickup_month", "pickup_day")
    .mode("overwrite")
    .parquet(DWS_OUT)
)
print(f"DWS written to: {DWS_OUT}")

# 验证
dws_check = spark.read.parquet(DWS_OUT)
print(f"DWS total rows: {dws_check.count():,}")
dws_check.groupBy("pickup_year", "pickup_month", "source_type") \
    .agg(F.sum("trip_count").alias("trips"), F.round(F.sum("gmv"), 0).alias("gmv")) \
    .orderBy("pickup_year", "pickup_month", "source_type").show()

spark.stop()
print("DWS build complete.")

26/10/19 14:57:40 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


DWS written to: /Users/lujiayi/tlc-spark-pipeline/data/warehouse/dws/trips_daily


DWS total rows: 559


+-----------+------------+-----------+------+-----------+
|pickup_year|pickup_month|source_type| trips|        gmv|
+-----------+------------+-----------+------+-----------+
|       2024|           3|     fhv_hv|739978|2.2096175E7|
|       2024|           3|      green|  1868|    42972.0|
|       2024|           3|     yellow|121063|  3401357.0|
+-----------+------------+-----------+------+-----------+



DWS build complete.
CPU times: user 19.4 ms, sys: 11.4 ms, total: 30.8 ms
Wall time: 2.79 s
